In [1]:
!python -m pip install --no-index --find-links=/kaggle/input/llm-classification-finetuning-model-loading/ -r /kaggle/input/llm-classification-finetuning-model-loading/requirements.txt

Looking in links: /kaggle/input/llm-classification-finetuning-model-loading/
Processing /kaggle/input/llm-classification-finetuning-model-loading/transformers-4.57.3-py3-none-any.whl (from -r /kaggle/input/llm-classification-finetuning-model-loading/requirements.txt (line 1))
Processing /kaggle/input/llm-classification-finetuning-model-loading/bitsandbytes-0.49.0-py3-none-manylinux_2_24_x86_64.whl (from -r /kaggle/input/llm-classification-finetuning-model-loading/requirements.txt (line 3))
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.1
    Uninstalling transformers-4.57.1:
      Successfully uninstalled transformers-4.57.1


In [2]:
import json
from dataclasses import dataclass
from typing import Dict, List, Any
import pandas as pd

from datasets import Dataset
from peft import PeftModel, PeftConfig
from transformers import AutoModel, AutoTokenizer, AutoConfig

from tqdm import tqdm
import torch
from torch.utils.data import DataLoader
import torch.nn.functional as F

2025-12-29 10:25:36.457915: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767003936.939905      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767003937.055242      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767003938.138468      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767003938.138512      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767003938.138515      24 computation_placer.cc:177] computation placer alr

In [3]:
# Max sequence length for input
MAX_SEQUENCE_LENGTH = 512

In [4]:
test_df = pd.read_csv("/kaggle/input/llm-classification-finetuning/test.csv")

In [5]:
LOCAL_ADAPTER_REPO = "/kaggle/input/llm-classification-finetuning-model-loading/deberta-v3-base-pairwise-sequence-classifier-peft-finetuned/"
LOCAL_MODEL_REPO = "/kaggle/input/llm-classification-finetuning-model-loading/deberta-v3-base-pairwise-sequence-classifier-base/"
LOCAL_BACKBONE_REPO = "/kaggle/input/llm-classification-finetuning-model-loading/deberta-v3-base/"
LOCAL_TOKENIZER_REPO = "/kaggle/input/llm-classification-finetuning-model-loading/deberta-v3-base-pairwise-sequence-classifier-tokenizer/"

adapter_config = PeftConfig.from_pretrained(LOCAL_ADAPTER_REPO, local_files_only=True)
tokenizer = AutoTokenizer.from_pretrained(LOCAL_TOKENIZER_REPO, local_files_only=True)
base_model_config = AutoConfig.from_pretrained(LOCAL_MODEL_REPO, trust_remote_code=True, local_files_only=True)

The module name  (originally ) is not a valid Python identifier. Please rename the original module to avoid import issues.
The module name  (originally ) is not a valid Python identifier. Please rename the original module to avoid import issues.


In [6]:
base_model_config.base_model_name = LOCAL_BACKBONE_REPO
base_model_config.base_model_name_or_path = LOCAL_BACKBONE_REPO

In [7]:
base_model = AutoModel.from_pretrained(LOCAL_MODEL_REPO, config=base_model_config, trust_remote_code=True, local_files_only=True)

The module name  (originally ) is not a valid Python identifier. Please rename the original module to avoid import issues.
The module name  (originally ) is not a valid Python identifier. Please rename the original module to avoid import issues.


In [8]:

# Load the Lora model
inference_model = PeftModel.from_pretrained(base_model, LOCAL_ADAPTER_REPO)

In [9]:
def safe_parse_json(x):
    if not isinstance(x, str):
        return x
    try:
        val = json.loads(x)
        # If it's a list, return first non-null element
        if isinstance(val, list):
            if val:
                return [item if item is not None else '' for item in val]
            else:
                return ''
        return val
    except json.JSONDecodeError:
        return ""
        
def format_conversation(query_list, response_list):
    parts = []
    for i, (q, r) in enumerate(zip(query_list, response_list)):
        parts.append((f"Query:\n{q}\n\nResponse:\n{r}"))
    return '\n\n'.join(parts)

def tokenize_pairwise(batch):
    a_encodings = tokenizer(
        batch["text_a"],
        padding="max_length",
        truncation=True,
        max_length=MAX_SEQUENCE_LENGTH,
    )
    b_encodings = tokenizer(
        batch["text_b"],
        padding="max_length",
        truncation=True,
        max_length=MAX_SEQUENCE_LENGTH,
    )

    out_dict = {
        "a_input_ids": a_encodings["input_ids"],
        "a_attention_mask": a_encodings["attention_mask"],
        "b_input_ids": b_encodings["input_ids"],
        "b_attention_mask": b_encodings["attention_mask"]
    }

    if "label" in batch:
        out_dict["labels"] = batch["label"]
    
    return out_dict

In [10]:
test_df["response_a_processed"] = test_df["response_a"].apply(safe_parse_json)
test_df["response_b_processed"] = test_df["response_b"].apply(safe_parse_json)
test_df["prompt_processed"] = test_df["prompt"].apply(safe_parse_json)

test_df['text_a'] = test_df.apply(lambda x: format_conversation(x['prompt_processed'], x['response_a_processed']), axis=1)
test_df['text_b'] = test_df.apply(lambda x: format_conversation(x['prompt_processed'], x['response_b_processed']), axis=1)

In [11]:
test_dataset = Dataset.from_pandas(test_df)

test_dataset = test_dataset.map(tokenize_pairwise, batched=True, remove_columns=list(test_df.columns))

test_dataset.set_format(
    type="torch",
    columns=["a_input_ids","a_attention_mask","b_input_ids","b_attention_mask"]
)

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

In [12]:
dataloader = DataLoader(
    test_dataset,
    batch_size=64,        # or whatever fits GPU
    shuffle=False
)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
inference_model.to(device)
inference_model.eval()

all_logits = []

for step, batch in enumerate(tqdm(dataloader)):
    # batch.to(device)
    batch = {k: v.to(device) for k, v in batch.items()}
    with torch.no_grad():
        outputs = inference_model(
            a_input_ids=batch["a_input_ids"],
            a_attention_mask=batch["a_attention_mask"],
            b_input_ids=batch["b_input_ids"],
            b_attention_mask=batch["b_attention_mask"],
        )

    all_logits.append(outputs['logits'].cpu())

logits = torch.cat(all_logits, dim=0)
preds = logits.argmax(dim=-1)
probs = F.softmax(logits, dim=-1)

100%|██████████| 1/1 [00:01<00:00,  1.93s/it]


In [13]:
test_df.loc[:, ['winner_model_a', 'winner_model_b', 'winner_tie']] = probs.numpy()

In [14]:
test_df[['id', 'winner_model_a', 'winner_model_b', 'winner_tie']].to_csv('submission.csv', index=False)